In [3]:
import numpy as np
import pandas as pd


Load Inputs (FCF + WACC):

In [4]:
fcf_df = pd.read_csv("../data/final_forecast_fcf.csv")
fcf_df


,Year,FCF_Mean,FCF_Upper,FCF_Lower
0,2025-12-31,1.183938e+12,1.444258e+12,9.236179e+11
1,2026-12-31,1.321301e+12,1.695406e+12,9.471964e+11
2,2027-12-31,1.464502e+12,1.946763e+12,9.822402e+11
3,2028-12-31,1.586764e+12,2.199054e+12,9.744737e+11
4,2029-12-31,1.710962e+12,2.450640e+12,9.712835e+11


In [5]:
wacc_df = pd.read_csv("../data/wacc_hdfc.csv")
wacc = wacc_df["wacc"].iloc[0]

wacc


np.float64(0.1105578784408892)

Prepare FCF Series:

In [6]:
fcf_values = fcf_df["FCF_Mean"].values
years = np.arange(1, len(fcf_values) + 1)

fcf_values, years


(array([1.18393796e+12, 1.32130138e+12, 1.46450171e+12, 1.58676401e+12,
        1.71096165e+12]),
 array([1, 2, 3, 4, 5]))

Discount Forecasted FCFs:

PV(FCFt​) = FCFt​​/((1+WACC)^t)

In [7]:
discounted_fcf = fcf_values / ((1 + wacc) ** years)

discounted_fcf


array([1.06607497e+12, 1.07132070e+12, 1.06921807e+12, 1.04315190e+12,
       1.01282471e+12])

Total PV of Explicit Forecast:

In [8]:
pv_forecast = discounted_fcf.sum()
pv_forecast


np.float64(5262590346378.274)

In [ ]:
Terminal Value (Gordon Growth Model):

In [9]:
terminal_growth_rate = 0.04  # 4%


TV = (FCFlast​×(1+g))/(WACC−g)​

In [10]:
terminal_fcf = fcf_values[-1]

terminal_value = (terminal_fcf * (1 + terminal_growth_rate)) / (wacc - terminal_growth_rate)

terminal_value


np.float64(25219013909429.293)

Discount Terminal Value to Present:

In [11]:
pv_terminal_value = terminal_value / ((1 + wacc) ** len(fcf_values))

pv_terminal_value


np.float64(14928704242587.896)

Enterprise Value (EV):

In [12]:
enterprise_value = pv_forecast + pv_terminal_value
enterprise_value


np.float64(20191294588966.17)

Wrap Everything into a DCF Function:

In [ ]:
def dcf_valuation(fcf, wacc, g):
    years = np.arange(1, len(fcf) + 1)
    pv_fcf = np.sum(fcf / ((1 + wacc) ** years))

    terminal_value = (fcf[-1] * (1 + g)) / (wacc - g)
    pv_terminal = terminal_value / ((1 + wacc) ** len(fcf))

    return pv_fcf + pv_terminal


test:

In [14]:
dcf_value = dcf_valuation(fcf_values, wacc, terminal_growth_rate)
dcf_value


np.float64(20191294588966.17)

Save DCF Output(for later use):

In [15]:
dcf_df = pd.DataFrame({
    "ticker": ["HDFCBANK.NS"],
    "enterprise_value": [dcf_value],
    "wacc": [wacc],
    "terminal_growth_rate": [terminal_growth_rate]
})

dcf_df


,ticker,enterprise_value,wacc,terminal_growth_rate
0,HDFCBANK.NS,2.019129e+13,0.110558,0.04


In [19]:
dcf_df.to_csv("../data/dcf_hdfc.csv", index=False)
dcf_value = dcf_df.loc[0, "enterprise_value"]

